# Forecast VAR, Source-Aware Edition

**Episode 2 story:** *I built an AI to predict the 2026 World Cup, then forced it to prove its sources.*

This notebook is the end-to-end experiment behind the YouTube episode. The drama is deliberately sports-shaped, but the lesson is agent engineering:

> A forecast agent is only useful if it can separate **football uncertainty** from **AI hallucination**.

In football, VAR means Video Assistant Referee. Here, Forecast VAR is the review-booth metaphor for AI forecasting: validate inputs, audit evidence, check claims, and refuse unsupported certainty before a prediction is presented.

The episode starts like a pundit panel: everyone wants a winner, every model has favourites, and a single upset can ruin the bracket. Then Forecast VAR steps in like a video-review booth. Before the agent is allowed to say who might win, it must prove five things:

1. the tournament field and feature table are internally valid,
2. the source coverage is disclosed,
3. forecast numbers come from deterministic tools rather than LLM vibes,
4. Monte Carlo probabilities are explained as simulations, not prophecy,
5. every final answer passes claim-level evaluation.

The workflow is:

1. Refresh a local evidence index.
2. Run forecast pre-flight validation.
3. Inspect source coverage.
4. Compare model output with a sample de-vig market baseline.
5. Run a whole-tournament Monte Carlo.
6. Demonstrate a rolling forecast after a completed-result state.
7. Evaluate the baseline and grounded agents.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from forecast_var.config import DEFAULT_OPENAI_MODEL
from forecast_var.tools import (
    refresh_evidence_index,
    preflight_forecast_context,
    source_coverage_report,
    forecast_match_with_context,
    simulate_tournament,
    rolling_group_forecast,
)
from forecast_var.eval_harness import evaluate
from forecast_var.mock_agent import run_baseline_mock, run_grounded_mock

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Default live OpenAI model:", DEFAULT_OPENAI_MODEL)

PROJECT_ROOT: /Users/mnajafi/Downloads/ai-reality-lab/wc26-forecast-var-agent-episode2
Default live OpenAI model: gpt-5.4-nano


## 0. Who predicts: `gpt-5.4-nano` or Monte Carlo?

Both appear in the project, but they do different jobs.

```text
gpt-5.4-nano
= live agent brain / router / tool caller / explainer

Python Monte Carlo simulator
= forecasting engine that generates probabilities through deterministic tools
```

In live mode, `gpt-5.4-nano` is the presenter in the studio. It reads the question, loads the right skills, calls MCP tools, and explains the output with caveats. The Monte Carlo simulator is the stats department. It repeatedly samples group and knockout outcomes, then counts how often each team becomes champion, finalist, or semifinalist.

This separation is the whole point of the episode. If the language model invents probabilities, the demo is flashy but unauditable. If the language model calls a tool, the numeric forecast can be reproduced when the code, bundled inputs, parameters, and simulator seed are the same. The live LLM API response itself is not guaranteed to be bit-for-bit deterministic, even with low temperature or seed controls. The tool boundary makes the prediction-producing part reproducible; the tool trace, citations, structured claims, and verifier make the LLM explanation auditable.

```text
User question
   ↓
gpt-5.4-nano Forecast VAR agent
   ↓
MCP tool call: simulate_tournament
   ↓
Python Monte Carlo engine
   ↓
probabilities
   ↓
gpt-5.4-nano explanation + citations + uncertainty
   ↓
claim verifier + eval harness
```

## 1. Build the local evidence index

The project does not use live APIs by default. Instead, it builds a transparent JSONL evidence index from bundled source cards, team features, sample market rows, and curated notes.

In [2]:
refresh = refresh_evidence_index()
refresh

{'output_path': 'data/generated/evidence_index.jsonl',
 'document_count': 73,
 'adapter_counts': {'source_cards': 15,
  'team_features': 48,
  'sample_market_odds': 7,
  'curated_evidence': 3},
 'include_live_api': False,
 'citations': [{'id': 'CITE-SRC-EVIDENCE-INDEX',
   'source_id': 'SRC-EVIDENCE-INDEX',
   'title': 'Generated evidence index',
   'url': 'file://data/generated/evidence_index.jsonl',
   'detail': 'Local evidence index built from source cards, curated notes, sample team features, and sample market odds. It powers transparent lightweight RAG for the forecast agent.'},
  {'id': 'CITE-SRC-SOURCE-REGISTRY',
   'source_id': 'SRC-SOURCE-REGISTRY',
   'title': 'Forecast VAR source registry',
   'url': 'file://data/sources/source_registry.json',
   'detail': 'Local registry describing which sources are bundled, refreshable, placeholders, manual or paid adapters, and disabled by default.'}]}

## 2. Forecast pre-flight validation

Before the agent is allowed to forecast, it must prove that the tournament field and model inputs are internally consistent.

In [3]:
preflight = preflight_forecast_context()
{
    "ready_for_forecast": preflight["ready_for_forecast"],
    "group_count": preflight["group_count"],
    "team_count": preflight["team_count"],
    "feature_rows": preflight["feature_rows"],
    "evidence_document_count": preflight["evidence_document_count"],
    "warnings": preflight["warnings"],
}

{'ready_for_forecast': True,
 'group_count': 12,
 'team_count': 48,
 'feature_rows': 48,
 'evidence_document_count': 73,
 'warnings': ['Forecasts use bundled demo priors unless adapters are refreshed.',
  'Do not forecast teams outside the validated 48-team field.',
  'Do not convert model probabilities into certainty or betting advice.']}

![Pre-flight readiness](../figures/preflight_readiness.png)

## 3. Source coverage report

A forecast should disclose its source coverage. Here the agent can distinguish official tournament structure from demo priors, sample market baselines, adapter slots, and missing live injury/lineup data.

In [4]:
coverage = source_coverage_report()
{
    "coverage": coverage["coverage"],
    "status_counts": coverage["status_counts"],
    "evidence_document_counts": coverage["evidence_document_counts"],
    "warnings": coverage["warnings"],
}

{'coverage': {'field_and_groups': 'covered_by_official_snapshot',
  'team_priors': 'bundled_demo_only',
  'market_baseline': 'bundled_demo_only_not_advice',
  'injuries_lineups': 'adapter_slot_or_curated_notes_only',
  'historical_calibration': 'adapter_slot_not_refreshed',
  'rolling_state': 'supported_when_results_are_supplied'},
 'status_counts': {'bundled_and_refreshable': 3,
  'bundled_crosscheck': 1,
  'adapter_placeholder': 4,
  'manual_or_paid_adapter': 1,
  'disabled_optional': 1,
  'bundled_demo_only': 1,
  'bundled_demo_model': 2,
  'adapter_placeholder_requires_key': 2,
  'fallback_adapter_placeholder': 1,
  'adapter_placeholder_public_research': 1,
  'bundled_demo_only_disabled_for_advice': 1,
  'generated_local': 1},
 'evidence_document_counts': {'SRC-API-FOOTBALL': 1,
  'SRC-CLUB-ELO': 1,
  'SRC-CURATED-EVIDENCE': 4,
  'SRC-EVIDENCE-INDEX': 1,
  'SRC-FIFA-RANKINGS': 1,
  'SRC-FIFA-WC26': 1,
  'SRC-FOOTBALL-DATA-ORG': 1,
  'SRC-SAMPLE-MARKET-ODDS': 8,
  'SRC-SAMPLE-TEAM-P

![Source coverage](../figures/source_coverage_status.png)

## 4. Model vs market baseline

Market data should not become betting advice. In Forecast VAR it is a comparison baseline: de-vig the sample odds, compare against the model, disclose margin and timestamp, then show data gaps.

De-vig means removing bookmaker margin, also called vig or overround. The tool converts each decimal odd into a raw implied probability with `1 / odds`, then normalizes the win/draw/win probabilities so they sum to 1. That gives a cleaner sample market-implied baseline for comparison, not a wagering recommendation or a claim about live licensed odds.

In [5]:
match_context = forecast_match_with_context("USA", "Australia")
{
    "model_probabilities": match_context["forecast"]["probabilities"],
    "market_baseline": match_context["market_baseline"],
    "model_vs_market": match_context["model_vs_market"],
    "data_gaps": match_context["data_gaps"],
}

{'model_probabilities': {'team_a': 'USA',
  'team_b': 'Australia',
  'p_team_a_win': 0.5088,
  'p_draw': 0.2099,
  'p_team_b_win': 0.2813,
  'rating_a_used': 1913.2,
  'rating_b_used': 1805.1},
 'market_baseline': {'match_id': 'G-D-USA-AUS',
  'team_a': 'USA',
  'team_b': 'Australia',
  'market_p_team_a_win': 0.5216,
  'market_p_draw': 0.2601,
  'market_p_team_b_win': 0.2183,
  'bookmaker_margin': 0.0533,
  'as_of_utc': '2026-05-16T00:00:00Z',
  'source_id': 'SRC-SAMPLE-MARKET-ODDS',
  'notes': 'Illustrative de-vig demo odds; not real betting advice.',
  'available': True,
  'warning': 'Illustrative demo market baseline only; not betting advice.',
  'citations': [{'id': 'CITE-SRC-SAMPLE-MARKET-ODDS',
    'source_id': 'SRC-SAMPLE-MARKET-ODDS',
    'title': 'Bundled sample market odds',
    'url': 'file://data/sources/sample_market_odds.csv',
    'detail': 'Illustrative 1X2 odds snapshot used to demonstrate de-vig market-implied probabilities and model-vs-market comparison. It is demo da

![Model vs market](../figures/model_vs_market_usa_australia.png)

## 5. Whole-tournament Monte Carlo

The upgraded agent can run a whole-tournament simulation. This is where the sports drama becomes useful for teaching agent design.

A deterministic bracket says: *Team A is stronger, so Team A advances.*

Monte Carlo says: *Team A is stronger, but football has variance.* The engine runs hundreds or thousands of alternate tournaments and then reports how often each team survives the chaos. That is why the final output is a probability distribution, not a prophecy. The simulator is deliberately documented as approximate: it models group advancement and a seeded 32-team knockout, not the official FIFA bracket path.

In [6]:
mc = simulate_tournament(sims=600, limit=8)
mc["top_teams"]

[{'team': 'Argentina',
  'group_winner': 0.7117,
  'round32': 0.9683,
  'round16': 0.7633,
  'quarterfinal': 0.67,
  'semifinal': 0.5283,
  'final': 0.355,
  'champion': 0.2167},
 {'team': 'France',
  'group_winner': 0.67,
  'round32': 0.965,
  'round16': 0.7317,
  'quarterfinal': 0.6067,
  'semifinal': 0.4433,
  'final': 0.2667,
  'champion': 0.1367},
 {'team': 'Spain',
  'group_winner': 0.6867,
  'round32': 0.9583,
  'round16': 0.69,
  'quarterfinal': 0.5583,
  'semifinal': 0.4017,
  'final': 0.235,
  'champion': 0.13},
 {'team': 'England',
  'group_winner': 0.64,
  'round32': 0.9383,
  'round16': 0.69,
  'quarterfinal': 0.525,
  'semifinal': 0.3567,
  'final': 0.22,
  'champion': 0.1183},
 {'team': 'Portugal',
  'group_winner': 0.6117,
  'round32': 0.95,
  'round16': 0.6933,
  'quarterfinal': 0.5317,
  'semifinal': 0.3017,
  'final': 0.1633,
  'champion': 0.0867},
 {'team': 'Brazil',
  'group_winner': 0.6233,
  'round32': 0.9467,
  'round16': 0.675,
  'quarterfinal': 0.485,
  'semif

![Monte Carlo champion probabilities](../figures/monte_carlo_champion_probabilities.png)

## 6. Rolling forecast after a completed-result state

Rolling mode locks completed results before forecasting the remaining path. The example below is intentionally illustrative, not a real 2026 result.

In [7]:
rolling = rolling_group_forecast(
    "D",
    [{"team_a": "USA", "team_b": "Australia", "team_a_goals": 2, "team_b_goals": 1}],
    sims=800,
)
rolling

{'group': 'D',
 'locked_results': [{'team_a': 'USA',
   'team_b': 'Australia',
   'team_a_goals': 2,
   'team_b_goals': 1}],
 'remaining_fixtures': [['USA', 'Paraguay'],
  ['USA', 'Türkiye'],
  ['Paraguay', 'Australia'],
  ['Paraguay', 'Türkiye'],
  ['Australia', 'Türkiye']],
 'probabilities': {'USA': {'winner': 0.6837, 'top2': 0.9062, 'top3': 0.9912},
  'Paraguay': {'winner': 0.1237, 'top2': 0.4088, 'top3': 0.7738},
  'Australia': {'winner': 0.0163, 'top2': 0.1588, 'top3': 0.435},
  'Türkiye': {'winner': 0.1762, 'top2': 0.5262, 'top3': 0.8}},
 'type': 'rolling_group_forecast',
 'warnings': ['Completed results are taken from the user-supplied JSON, not from a live results feed.',
  'Remaining match probabilities still use demo priors unless source adapters are refreshed.'],
 'citations': [{'id': 'CITE-SRC-FIFA-WC26',
   'source_id': 'SRC-FIFA-WC26',
   'title': 'FIFA World Cup 2026 official tournament page',
   'url': 'https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa20

## 7. Agent answer examples

The deterministic grounded agent follows the same workflow a live OpenAI/MCP agent is instructed to follow: source search, pre-flight, tool call, typed claims, citations, and claim verification. The live version uses `gpt-5.4-nano` by default, but the offline notebook uses deterministic mock mode so the episode is reproducible without API calls.

For live API runs, treat reproducibility as protocol reproducibility rather than identical prose: preserve the model name, prompt, tool inputs, tool outputs, code version, data snapshot, and simulator seed, then compare the structured answer and claim verification.

In [8]:
questions = [
    "Give me a source coverage report before using the forecast agent for public predictions.",
    "Compare the model against the sample market baseline for USA vs Australia.",
    "Run a Monte Carlo tournament simulation and show the top champion probabilities.",
    "After USA beat Australia 2-1, run a rolling Group D forecast.",
]

for q in questions:
    ans = run_grounded_mock(q)
    print("QUESTION:", q)
    print("ANSWER:", ans.answer)
    print("TOOLS:", ans.tools_used)
    print("SKILLS:", ans.skills_used)
    print("PROBABILITIES:", ans.probabilities)
    print("-" * 100)

QUESTION: Give me a source coverage report before using the forecast agent for public predictions.
ANSWER: Source coverage report: field/groups are covered by the official snapshot; team priors and sample market baselines are bundled demo inputs; injuries and lineups are adapter slots or curated notes only; rolling forecasts are supported when completed results are supplied. Evidence document counts by source include {'SRC-API-FOOTBALL': 1, 'SRC-CLUB-ELO': 1, 'SRC-CURATED-EVIDENCE': 4, 'SRC-EVIDENCE-INDEX': 1, 'SRC-FIFA-RANKINGS': 1, 'SRC-FIFA-WC26': 1, 'SRC-FOOTBALL-DATA-ORG': 1, 'SRC-SAMPLE-MARKET-ODDS': 8, 'SRC-SAMPLE-TEAM-PRIORS': 49, 'SRC-STATSBOMB-HISTORICAL-WC': 1, 'SRC-WIKI-WC26': 1}.
TOOLS: ['search_source_cards', 'preflight_forecast_context', 'source_coverage_report', 'verify_claims_against_sources']
SKILLS: ['forecast-preflight', 'citation-discipline', 'source-triage', 'forecast-modeling', 'uncertainty-calibration', 'source-adapter-governance']
PROBABILITIES: {}
------------

QUESTION: Run a Monte Carlo tournament simulation and show the top champion probabilities.
ANSWER: The Monte Carlo demo ran 500 simulations. Top champion probabilities are Argentina 21.0%, France 14.2%, Spain 14.0%, England 12.0%, Portugal 8.4%. The simulator approximates a seeded 32-team knockout and should be treated as an educational model, not a certainty.
TOOLS: ['search_source_cards', 'preflight_forecast_context', 'simulate_tournament', 'verify_claims_against_sources']
SKILLS: ['forecast-preflight', 'citation-discipline', 'forecast-modeling', 'uncertainty-calibration', 'monte-carlo-forecasting']
PROBABILITIES: {'Argentina_champion': 0.21, 'France_champion': 0.142, 'Spain_champion': 0.14, 'England_champion': 0.12, 'Portugal_champion': 0.084}
----------------------------------------------------------------------------------------------------
QUESTION: After USA beat Australia 2-1, run a rolling Group D forecast.
ANSWER: Rolling Group D forecast after locking the illustrative result

### Live OpenAI path used in the episode examples

The live path keeps the same architecture but swaps the deterministic mock narrator for the OpenAI Agents SDK agent. The default model is `gpt-5.4-nano`.

```bash
export OPENAI_API_KEY="your_key_here"

PYTHONPATH=src python scripts/run_agent.py   "Run a Monte Carlo tournament simulation and show the top champion probabilities."   --mode openai
```

You can still override the model explicitly:

```bash
PYTHONPATH=src python scripts/run_agent.py   "Compare the model against the sample market baseline for USA vs Australia."   --mode openai   --model gpt-5.4-nano
```


## 8. Evaluation harness

The baseline is intentionally careless: no tools, no pre-flight, no citations, no uncertainty discipline. The grounded agent must pass the same cases with explicit source support.

In [9]:
# The scripts already materialise the evaluation summaries in reports/.
# Reading them here keeps the notebook fast and makes the episode reproducible.
baseline = json.loads((PROJECT_ROOT / "reports/summary_baseline_mock.json").read_text())
grounded = json.loads((PROJECT_ROOT / "reports/summary_grounded_mock.json").read_text())
{"baseline": baseline, "grounded": grounded}

{'baseline': {'cases': 14,
  'pass_rate': 0.0,
  'tool_recall': 0.0,
  'skill_recall': 0.0,
  'preflight_recall': 0.0,
  'citation_recall': 0.0,
  'source_support_precision': 0.0,
  'factual_citation_support': 1.0,
  'model_citation_support': 1.0,
  'unsupported_factual_claim_rate': 0.0,
  'unsupported_prediction_claim_rate': 0.0,
  'unsupported_claim_rate': 1.0,
  'abstention_accuracy': 0.7142857142857143,
  'probability_sanity_rate': 0.42857142857142855},
 'grounded': {'cases': 14,
  'pass_rate': 1.0,
  'tool_recall': 1.0,
  'skill_recall': 1.0,
  'preflight_recall': 1.0,
  'citation_recall': 1.0,
  'source_support_precision': 1.0,
  'factual_citation_support': 1.0,
  'model_citation_support': 1.0,
  'unsupported_factual_claim_rate': 0.0,
  'unsupported_prediction_claim_rate': 0.0,
  'unsupported_claim_rate': 0.0,
  'abstention_accuracy': 1.0,
  'probability_sanity_rate': 1.0}}

![Evaluation summary](../figures/eval_summary.png)

## Takeaway

The prediction itself is not the point. The agent-engineering lesson is that a credible forecast agent needs a hard boundary between **LLM orchestration** and **statistical forecasting**:

- source coverage disclosure,
- source-specific adapters,
- reproducible evidence indexing,
- seeded deterministic model tools,
- market-baseline comparison without betting advice,
- rolling-state support,
- typed claims and citations,
- evaluation beyond “the answer looked good”.